# Deep Learning Aiyagari Solver (Full Working Notebook)
This notebook contains the full implementation of the deep-learning-based Aiyagari solver.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

beta = 0.96
sigma = 2.0
r = 0.04
w = 1.0

a_min = 0.0
a_max = 50.0

z_states = np.array([0.5, 1.5], dtype=np.float32)
P = np.array([[0.9, 0.1],
              [0.1, 0.9]], dtype=np.float32)

def sample_next_z(z_idx):
    z_idx = np.asarray(z_idx, dtype=int)
    probs = P[z_idx]
    draws = np.random.rand(z_idx.shape[0])
    next_idx = (draws > probs[:, 0]).astype(int)
    return next_idx


In [ ]:

class PolicyNet(nn.Module):
    def __init__(self, hidden_dim=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, a, z):
        x = torch.cat([a, z], dim=1)
        a_next = self.net(x)
        a_next = torch.nn.functional.softplus(a_next)
        return torch.clamp(a_next, a_min, a_max)

policy_net = PolicyNet().to(device)
policy_net


In [ ]:

def marginal_utility(c):
    eps = 1e-8
    return torch.clamp(c, eps).pow(-sigma)

def euler_residual(a, z):
    y = w * z + (1 + r) * a
    a_next = policy_net(a, z)
    c = y - a_next
    mu_c = marginal_utility(c)

    z_idx = (z.detach().cpu().numpy().reshape(-1) > 1.0).astype(int)
    z_next_idx = sample_next_z(z_idx)
    z_next_vals = z_states[z_next_idx]

    a_next_detached = a_next.detach().cpu().numpy().reshape(-1)
    a_next_t = torch.tensor(a_next_detached, dtype=torch.float32, device=device).view(-1, 1)
    z_next_t = torch.tensor(z_next_vals, dtype=torch.float32, device=device).view(-1, 1)

    y_next = w * z_next_t + (1 + r) * a_next_t
    a_next2 = policy_net(a_next_t, z_next_t)
    c_next = y_next - a_next2
    mu_c_next = marginal_utility(c_next)

    rhs = beta * (1 + r) * mu_c_next
    residual = mu_c - rhs
    return residual, c, c_next

def loss_batch(batch_size=2048):
    a = torch.rand(batch_size, 1, device=device) * a_max
    z_idx = np.random.randint(0, 2, size=batch_size)
    z_vals = z_states[z_idx]
    z = torch.tensor(z_vals, dtype=torch.float32, device=device).view(-1, 1)

    residual, c, c_next = euler_residual(a, z)
    loss_euler = (residual ** 2).mean()

    penalty_c = (torch.relu(-c + 1e-3) ** 2).mean()
    penalty_c_next = (torch.relu(-c_next + 1e-3) ** 2).mean()

    loss = loss_euler + 10.0 * (penalty_c + penalty_c_next)
    return loss, float(loss_euler.detach().cpu().item())


In [ ]:

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)

num_epochs = 200
print_every = 20

loss_history = []
euler_history = []

for epoch in range(1, num_epochs + 1):
    optimizer.zero_grad()
    loss, euler_loss_val = loss_batch(batch_size=4096)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    euler_history.append(euler_loss_val)

    if epoch % print_every == 0 or epoch == 1:
        print(f"Epoch {epoch:4d} | total loss = {loss.item():.5f} | Euler MSE = {euler_loss_val:.5f}")


In [ ]:

plt.figure()
plt.plot(loss_history, label="Total loss")
plt.plot(euler_history, label="Euler residual MSE")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()


In [ ]:

a_grid = np.linspace(a_min, a_max, 100, dtype=np.float32)
a_t = torch.tensor(a_grid, dtype=torch.float32, device=device).view(-1, 1)

plt.figure()
for z_val in z_states:
    z_t = torch.full_like(a_t, z_val)
    with torch.no_grad():
        a_next_t = policy_net(a_t, z_t)
    a_next = a_next_t.cpu().numpy().reshape(-1)
    plt.plot(a_grid, a_next, label=f"z = {z_val:.1f}")

plt.xlabel("Current assets a")
plt.ylabel("Next-period assets a_next")
plt.title("Learned savings policy a_next = f(a,z)")
plt.legend()
plt.show()


In [ ]:

T_sim = 200
N_agents = 5000

a_agents = np.zeros((N_agents,), dtype=np.float32)
z_idx_agents = np.zeros((N_agents,), dtype=int)

mean_assets_path = []
gini_path = []

def gini_coefficient(x):
    x = np.sort(x)
    n = len(x)
    cumx = np.cumsum(x)
    if cumx[-1] == 0:
        return 0.0
    B = cumx.sum() / (cumx[-1] * n)
    return 1 + 1/n - 2*B

for t in range(T_sim):
    a_t = torch.tensor(a_agents, dtype=torch.float32, device=device).view(-1, 1)
    z_vals = z_states[z_idx_agents]
    z_t = torch.tensor(z_vals, dtype=torch.float32, device=device).view(-1, 1)

    with torch.no_grad():
        y_t = w * z_t + (1 + r) * a_t
        a_next = policy_net(a_t, z_t)
        c_t = y_t - a_next

    a_agents = a_next.cpu().numpy().reshape(-1)
    a_agents = np.clip(a_agents, a_min, a_max)

    z_idx_agents = sample_next_z(z_idx_agents)

    mean_assets_path.append(a_agents.mean())
    gini_path.append(gini_coefficient(a_agents))

fig, ax = plt.subplots(2, 1, figsize=(6, 6))

ax[0].plot(mean_assets_path)
ax[0].set_ylabel("Mean assets")
ax[0].set_title("Mean assets over time")

ax[1].plot(gini_path)
ax[1].set_ylabel("Gini coefficient")
ax[1].set_xlabel("Time")
ax[1].set_title("Wealth inequality (Gini) over time")

plt.tight_layout()
plt.show()
